# 프롬프트(시스템 역할 메시지)를 통한 챗봇 구현


## OpenAI LLM 준비


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# .env 로드
load_dotenv()

# LLM 초기화
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

## 프롬프트 메시지 구성


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 '명탐정 코난' 전문가입니다. 모든 질문에 친절하고 정확하게 답해주세요.",
        ),
        ("human", "{question}"),
    ]
)

## Parser를 통한 LLM 응답 처리 함수

사용자 입력을 LLM에 전달하고 `StrOutputParser`를 통한 텍스트 응답 처리


In [ ]:
from langchain_core.output_parsers import StrOutputParser


def response_from_llm(user_input: str) -> str:
    """사용자 입력을 기반으로 LLM에서 응답을 생성합니다."""

    # 사용자 입력(user_input)을 이용해 프롬프트 메시지 구성
    prompt_messages = prompt.format_messages(question=user_input)

    # LLM으로 프롬프트 메시지 전달하여 응답 수신
    ai_message = llm.invoke(prompt_messages)

    # LLM 전달 메시지 파서
    parser = StrOutputParser()

    # UI로 출력할 content 부분만 전달 invoke 사용 - parser.parse(ai_message.content)
    return parser.invoke(ai_message)

## Gradio 인터페이스 구성


In [ ]:
import gradio as gr


# Gradio 챗봇 이벤트 핸들러 함수
def show_chat_dialog(
    user_input: str, chat_history: list[dict[str, str]]
) -> tuple[list[dict[str, str]], str]:

    if user_input and user_input.strip():
        chat_history += [{"role": "user", "content": user_input}]
        response = response_from_llm(user_input)
        chat_history += [{"role": "assistant", "content": response}]
    else:
        chat_history += [{"role": "assistant", "content": "질문을 입력해 주세요."}]

    return chat_history, ""


# Gradio UI 구성
with gr.Blocks() as demo:

    chatbot = gr.Chatbot(label="🕵️‍♂️ 명탐정 코난 세계관 챗봇", height=200)
    user_input = gr.Textbox(label="궁금한 점을 입력하세요")

    # 버튼 이벤트 연결
    user_input.submit(
        fn=show_chat_dialog,
        inputs=[user_input, chatbot],
        outputs=[chatbot, user_input],
    )

demo.launch()

In [ ]:
demo.close()